# Lesson 1.7: Ensemble oethods

*Module 1 · oath Prerequisites*

> Random Forest, XGBoost, LightGBo, CatBoost, StackingClassifier, VotingClassifier — and how model routing is stacking for the GenAI era.

`New Module` · `5 Concepts` · `All Runnable` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-1.7.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 2 — Bagging & Random Forest — `random_forest.py`
2. Step 3 — Boosting — The Holy Trinity — `boosting_comparison.py`
3. Step 4 — Stacking & Voting — `stacking_voting.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q scikit-learn datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 2 · Bagging & Random Forest

100 trees vote independently

**`random_forest.py`** · _Block 1 of 3_


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=500, max_features='sqrt',
                            n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
print(f"RF Accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}")
print(f"Top-3 features: {rf.feature_importances_.argsort()[-3:][::-1]}")


> **What just happened?** 500 trees each trained on a random ~63% of the data (bootstrap) with random feature subsets. Their votes are aggregated by majority. feature_importances_ shows which features matter most — this is explainability . In Indian BFSI, RF is often the first model tried because regulators require interpretability.


### Step 3 · Boosting — The Holy Trinity

XGBoost, LightGBo, CatBoost

**`boosting_comparison.py`** · _Block 2 of 3_


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# sklearn's GradientBoosting (XGBoost-like behavior)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                max_depth=3, random_state=42)
start = time.time()
gb.fit(X_tr, y_tr)
elapsed = time.time() - start
acc = accuracy_score(y_te, gb.predict(X_te))
print(f"GradientBoosting: accuracy={acc:.4f}, time={elapsed:.2f}s")


> **What just happened?** 200 trees built sequentially, each focusing on the errors of the previous ensemble. learning_rate=0.1 controls how much each tree contributes (lower = more trees needed but better generalization). XGBoost/LightGBo dominate Kaggle and Indian BFSI — they’re the default for tabular data until proven otherwise.


### Step 4 · Stacking & Voting

Combining diverse models

**`stacking_voting.py`** · _Block 3 of 3_


In [ ]:
from sklearn.ensemble import (StackingClassifier, VotingClassifier,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Base models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
svc = SVC(probability=True, random_state=42)

# Voting: all models vote
voter = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('svc', svc)],
    voting='soft'  # average probabilities
)

# Stacking: meta-learner decides
stacker = StackingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('svc', svc)],
    final_estimator=LogisticRegression(),
    cv=5  # out-of-fold predictions
)

voter.fit(X_tr, y_tr)
stacker.fit(X_tr, y_tr)
print(f"Voting   accuracy: {accuracy_score(y_te, voter.predict(X_te)):.4f}")
print(f"Stacking accuracy: {accuracy_score(y_te, stacker.predict(X_te)):.4f}")


> **What just happened?** The VotingClassifier averages all three models’ probabilities (soft voting). The StackingClassifier goes further — it trains a LogisticRegression on 5-fold out-of-fold predictions from the base models. The meta-learner learns which model to trust for which type of input. This is the classical version of what LLM routing does.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
